In [31]:
import pandas as pd

# 1. Load data
df_all = pd.read_csv("results.csv")

# 2. Beddel l-date l format s7i7
df_all['date'] = pd.to_datetime(df_all['date'])

# 3. N-khelliw gha l-matchat men 2018 l-fouq (Bach l-model i-koun modern)
df = df_all[df_all['date'].dt.year >= 2018].copy()

# 4. Filtri gha l-matchat li fihom l-Maroc
df = df[(df['home_team'] == 'Morocco') | (df['away_team'] == 'Morocco')].copy()

# 5. Creyi l-'target' (1 ila rbe7 l-Maroc, 0 ila khser wala t3adel)
def get_result(row):
    if row['home_team'] == 'Morocco':
        return 1 if row['home_score'] > row['away_score'] else 0
    else:
        return 1 if row['away_score'] > row['home_score'] else 0

df['target'] = df.apply(get_result, axis=1)

print(f"Data ready! Rows found for Morocco: {len(df)}")
df.head()

Data ready! Rows found for Morocco: 116


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,target
41329,2018-03-23,Serbia,Morocco,1.0,2.0,Friendly,Turin,Italy,True,1
41398,2018-03-27,Morocco,Uzbekistan,2.0,0.0,Friendly,Casablanca,Morocco,False,1
41469,2018-05-31,Ukraine,Morocco,0.0,0.0,Friendly,Geneva,Switzerland,True,0
41535,2018-06-04,Slovakia,Morocco,1.0,2.0,Friendly,Geneva,Switzerland,True,1
41587,2018-06-09,Estonia,Morocco,1.0,3.0,Friendly,Tallinn,Estonia,False,1


In [32]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import joblib
import pandas as pd

# 1. Goal Difference (Corrected)
df['goal_difference'] = (df['home_score'] - df['away_score']).abs()

# 2. Logic dial l-Form (Akher 5 matchat)
def get_form(df_in):
    df_sorted = df_in.sort_values('date')
    return df_sorted['target'].rolling(window=5, min_periods=1).mean()

df['form_score'] = get_form(df)

# 3. Strength Level mapping
strong_teams = ['France', 'Spain', 'Brazil', 'Argentina', 'Portugal', 'England', 'Belgium', 'Netherlands']
df['opponent'] = df.apply(lambda x: x['away_team'] if x['home_team'] == 'Morocco' else x['home_team'], axis=1)
df['opponent_strength_level'] = df['opponent'].apply(lambda x: 'Strong' if x in strong_teams else 'Medium')

# 4. Competition Type
df['competition_type'] = df['tournament'].apply(lambda x: 'Official' if x in ['FIFA World Cup', 'African Cup of Nations', 'FIFA World Cup qualification'] else 'Friendly')

# 5. One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_cols = ['competition_type', 'opponent_strength_level']
encoded_data = encoder.fit_transform(df[cat_cols])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(cat_cols), index=df.index)

# 6. Final Features Selection
# Zidna hna 'goal_difference' bach l-model i-welli kiy-fhem l-power d l-hujum
num_cols = ['form_score', 'goal_difference'] 
X = pd.concat([df[num_cols], encoded_df], axis=1)
y = df['target']

print("✅ Features processed successfully!")
X.head()

✅ Features processed successfully!


,form_score,goal_difference,competition_type_Friendly,competition_type_Official,opponent_strength_level_Medium,opponent_strength_level_Strong
41329,1.000000,1.0,1.0,0.0,1.0,0.0
41398,1.000000,2.0,1.0,0.0,1.0,0.0
41469,0.666667,0.0,1.0,0.0,1.0,0.0
41535,0.750000,1.0,1.0,0.0,1.0,0.0
41587,0.800000,2.0,1.0,0.0,1.0,0.0


In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Split l-data (Corrected parameter name)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Cree o Train l-Model
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_rf.fit(X_train, y_train)

# 3. Check l-Accuracy
y_pred = model_rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {acc*100:.2f}%")

# 4. Save the files for Streamlit
joblib.dump(model_rf, 'morocco_wc_model.pkl')
joblib.dump(encoder, 'encoder.pkl')

print("\n🚀 Files are ready! You can now run the Streamlit app.")

✅ Model Accuracy: 95.83%

🚀 Files are ready! You can now run the Streamlit app.
